# Acceptance: the calibration workflow, end to end

**This is a sign-off page, not a tutorial.** It exercises roadmap step 28, the
last step of the calibration workflow toolkit (coverage-ledger rows `CW10` and
`CW11`): what a modeller looks at after calibrating. `bm.posterior_runs` turns
the points any stage produced into forward runs, and six figures draw them:

| Figure | Draws |
|---|---|
| `wf.plot_design(candidates)` | log density against each parameter |
| `wf.plot_optimisation(run)` | best loss per start of an `OptimizeRun` |
| `wf.plot_chains(run)` | per-chain traces of an `MCMCRun`, R-hat and ESS in each panel title |
| `wf.plot_spaghetti(runs, output)` | one line per sampled trajectory |
| `wf.plot_ribbons(runs, output)` | quantile bands over the draws, one colour per scenario |
| `wf.plot_scenarios(runs, output, ref=)` | each scenario's paired difference from a reference |

Every figure is an ordinary Plotly figure, takes `fig=` to draw onto one you
already have, and `targets=` (or `wf.add_targets`) overlays the calibration
data as black dots. Nothing here asserts; the automated checks live in
`tests/test_calib_workflow_plots.py`.

The model is notebook 24's: **many quite different parameter sets fit the data
almost equally well**, and they disagree about what happens next. That is what
makes the figures worth looking at — with one good fit, every line would be the
same line.

Run every cell top to bottom (about three minutes; section D compiles and runs
NUTS). Sign off only if every **What to check** box holds.

| | You are checking that… |
|---|---|
| A | the design figure shows where the good fits lie, and the best points sit on top of it |
| B | *LHS → best → runs*: the design's best points, as spaghetti, pass through the data and then disagree |
| C | *optimise → runs*: the optimisation converged, and its fits hug the data more tightly than the design's — but still disagree beyond it |
| D | *the full chain*: seeded MCMC converged, and the posterior band passes through the data and fans out beyond it |
| E | scenario outputs: a distancing scenario plotted against the baseline, and its effect as a paired difference |
| F | the figures compose on one canvas, and each is a few lines over data you can reach yourself |

## The model: a ridge of good fits

An SIR epidemic in a population of one million with frequency-dependent
transmission. Two rates are unknown: the **contact rate** and the **recovery
rate**. The data are infectious counts every four days over the first four
weeks only — the growth phase — with negative-binomial noise. Early growth is
exponential at rate *contact − recovery*, so the data pin down that difference
but hardly the two rates separately.

One addition to notebook 24, used only in section E: the contact rate is
multiplied by `1 − distancing` from day 35. The fit holds `distancing` at 0, so
the fitted model is exactly notebook 24's; a scenario sets it to 0.4.

The figure is the data, drawn with `wf.add_targets` on an empty figure.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from numpyro.infer import NUTS

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    Time,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import BayesianModel, NegativeBinomial, Scenario, Uniform
from summer4.epi.calibration import workflow as wf
from summer4.timevarying import step

state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)
model = FlowModel(pmap)
distancing = step(Time(), [35.0], [1.0, 1.0 - Param("distancing")])
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=pop,
            mixing=MixingMatrix(pop, np.array([[1.0]])),
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("contact") * distancing,
        ),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
cm = model.compile()

N = 1.0e6
y0_values = np.zeros(pmap.size)
y0_values[pmap.select(state["S"])] = N - 20
y0_values[pmap.select(state["I"])] = 20
y0 = PropertyData.wrap(pmap, y0_values)
infectious = Compartments(where=state["I"])
run_kwargs = dict(t0=0.0, t1=180.0, dt=0.5, solver="euler")
TRUTH = {"contact": 0.45, "recovery": 0.2, "distancing": 0.0}

times = np.arange(0.0, 29.0, 4.0)
clean = cm.run(TRUTH, y0, save=SavePlan(requests={"I": SaveRequest(infectious, ts=times)}), **run_kwargs)
expected = np.asarray(clean["I"].values.data).reshape(-1)
observed = np.random.default_rng(3).negative_binomial(20, 20 / (20 + expected)).astype(float)

bm = BayesianModel(
    cm,
    {"distancing": 0.0},
    priors=(Uniform("contact", 0.1, 1.5), Uniform("recovery", 0.02, 1.0)),
    targets=TargetSet(
        targets=(Target(key="I", times=times, values=observed, quantity=infectious, likelihood=NegativeBinomial(dispersion=20.0)),)
    ),
    y0=y0,
    run_kwargs=run_kwargs,
)

fig = wf.add_targets(go.Figure(), bm.targets, name="observed infectious")
fig.update_layout(title="The data: infectious people, the first four weeks only", xaxis_title="day", yaxis_title="infectious")
fig.show()

> **What to check**
>
> - Eight black dots from day 0 to day 28, rising from about 20 to a few hundred.

## A. The design, and its best points

Score a 256-point Latin hypercube over the priors and keep the sixteen best.
`plot_design` draws log density against each parameter, one panel per
parameter; passing the figure back with `fig=` overlays the best sixteen in
the same panels. The y axis is cut 60 below the best point — the worst design
points are far lower. The table lists the best sixteen with *contact −
recovery* (the early growth rate the data measure) and R0 (contact ÷ recovery).

In [ ]:
design = wf.evaluate(bm, wf.lhs(bm, 256, seed=0), batch_size=128)
best = design.best(16)

fig = wf.plot_design(design, name="256-point design")
wf.plot_design(best, fig=fig, name="best 16")
top = float(np.max(best.log_density))
fig.update_yaxes(range=[top - 60, top + 5])
fig.show()

table_a = best.to_frame()
table_a["growth rate"] = table_a["contact"] - table_a["recovery"]
table_a["R0"] = table_a["contact"] / table_a["recovery"]
table_a.round(3)

> **What to check**
>
> - Two panels, *contact* and *recovery*, sharing a *log density* axis; the figure title names the design.
> - The *best 16* markers (second colour) are the top of the cloud in **both** panels.
> - They are spread across a wide range of each rate — the design has not pinned either one down — yet in the table every growth rate is close to 0.25 while R0 varies a lot.

## B. LHS → best → runs

The shortest composition: hand the design's best points straight to
`posterior_runs`. The projection output is named `I`, like the target, so
`targets=bm.targets` finds the data to overlay. Two scenarios are run for every
set of points in this notebook — the baseline and the distancing scenario of
section E — and each figure picks the one it draws. The dashed line is the end
of the data. The y axis is logarithmic from 10 to a million (a line added to the
returned figure, and used again below), so the data weeks and the peaks weeks
later are both visible.

In [ ]:
projection = OutputSet()
projection["I"] = Compartments(where=state["I"]).total()
scenarios = {
    "baseline": Scenario(outputs=projection),
    "distancing": Scenario(params={"distancing": 0.4}, outputs=projection),
}

design_runs = bm.posterior_runs(best, n=None, scenarios=scenarios)
fig = wf.plot_spaghetti(design_runs, "I", n=None, targets=bm.targets)
fig.add_vline(x=28, line_dash="dash", annotation_text="end of data")
fig.update_layout(xaxis_title="day")
fig.update_yaxes(type="log", range=[1, 6])
fig.show()

peaks = design_runs.samples("baseline", "I").max(axis=1)
pd.Series({"trajectories": design_runs.n, "lowest peak": round(peaks.min()), "highest peak": round(peaks.max())}, name="B")

> **What to check**
>
> - One legend entry, *baseline: 16 draws*, plus *target I*; the title reads *I: 16 sampled trajectories (baseline)*.
> - Left of the dashed line the sixteen lines run through the cloud of black dots; to the right they peak at clearly different heights and days (the table's highest peak is several times the lowest).

## C. Optimise → runs

Optimise from the best sixteen with `wf.optimize` (Adam, automatic
learning-rate probe and convergence), then hand the `OptimizeRun` itself to
`posterior_runs`, which uses each start's best point. The second figure layers
the optimised fits over the design's (in grey) on one figure with `fig=`.

In [ ]:
fitted = wf.optimize(bm, best, max_steps=300, seed=0)
wf.plot_optimisation(fitted).show()

fitted_runs = bm.posterior_runs(fitted, n=None, scenarios=scenarios)
fig = wf.plot_spaghetti(design_runs, "I", n=None, color="lightgrey", opacity=1.0, name="design's best 16")
wf.plot_spaghetti(fitted_runs, "I", n=None, fig=fig, name="optimised 16", targets=bm.targets)
fig.add_vline(x=28, line_dash="dash", annotation_text="end of data")
fig.update_layout(title="C. Optimised fits (colour) over the design's best points (grey)", xaxis_title="day")
fig.update_yaxes(type="log", range=[1, 6])
fig.show()

data_days = np.searchsorted(fitted_runs.times, times)
fit_error = lambda runs: float(np.abs(np.log(runs.samples("baseline", "I")[:, data_days] / observed)).mean())
pd.DataFrame(
    {
        "best log density": [float(np.max(best.log_density)), float(np.max(fitted.candidates.log_density))],
        "worst log density": [float(np.min(best.log_density)), float(np.min(fitted.candidates.log_density))],
        "mean |log(fit / data)| over the data days": [fit_error(design_runs), fit_error(fitted_runs)],
        "spread of projected peaks (max/min)": [
            float(runs.samples("baseline", "I").max(axis=1).max() / runs.samples("baseline", "I").max(axis=1).min())
            for runs in (design_runs, fitted_runs)
        ],
    },
    index=["design's best 16", "optimised 16"],
).round(3)

> **What to check**
>
> - Optimisation figure: sixteen lines, one per start, falling and then flattening; the title counts the converged starts.
> - Layered figure: the coloured optimised lines sit closer together through the data than the grey design lines, but still fan out after the dashed line.
> - Table: the optimised worst log density is better than the design's worst, and the mean log distance from the data is smaller — but the highest projected peak is still many times the lowest. Optimisation sharpens the fit; it does not settle the projection.

## D. The full chain: seeded MCMC → runs

Four NUTS chains seeded from the four best optimised points, sampling in
chunks of 100 draws until `wf.StopRule(rhat=1.05, ess=100)` is met.
`plot_chains` shows each chain's trace per parameter, with split R-hat and bulk
ESS in each panel title and a dotted line between chunks. Then 200 posterior
draws are run forward and drawn as quantile ribbons (2.5–97.5% and 25–75%
bands, and the median), on the same logarithmic axis.

In [ ]:
post = bm.sample(
    NUTS(bm.numpyro_model(), target_accept_prob=0.95),
    init=fitted.candidates.best(4),
    num_chains=4,
    num_warmup=200,
    num_samples=100,
    stop=wf.StopRule(rhat=1.05, ess=100, max_samples=1500),
    seed=0,
)
wf.plot_chains(post).show()

post_runs = bm.posterior_runs(post, n=200, seed=1, batch_size=100, scenarios=scenarios)
fig = wf.plot_ribbons(post_runs, "I", scenarios=["baseline"], targets=bm.targets)
fig.add_vline(x=28, line_dash="dash", annotation_text="end of data")
fig.update_layout(xaxis_title="day")
fig.update_yaxes(type="log", range=[1, 6])
fig.show()

band = post_runs.quantiles(q=(0.025, 0.975))["baseline"]
width = band[("I", "0.975")] - band[("I", "0.025")]
pd.Series(
    {
        "stop reason": post.reason,
        "chunks": post.chunks,
        "95% band width at day 28": round(float(width.loc[28.0])),
        "95% band width at its widest": round(float(width.max())),
    },
    name="D",
    dtype=object,
)

> **What to check**
>
> - Chains figure: two panels (*contact*, *recovery*), four chain colours mixed together in each rather than each in its own band; every panel title shows R-hat ≤ 1.05 and bulk ESS ≥ 100; the figure title reads *stop: diagnostics*, and there is one dotted line fewer than the number of chunks.
> - Ribbon figure: up to the dashed line the band is narrow and the median runs through the cloud of dots; after it the band fans out widely. (The band is the spread of model trajectories, not of noisy observations, so individual dots may sit just outside it.)
> - The table's widest band is many times its width at day 28.

## E. Scenario outputs against a reference

The distancing scenario cuts the contact rate by 40% from day 35. The first
figure draws both scenarios' ribbons (95% band and median). The second is
`plot_scenarios`: for every posterior draw, distancing minus baseline, with the
**same** parameters on both sides — so the band is the uncertainty in the
*effect*, not the overlap of two independent clouds. Below the dashed zero line
means fewer infectious people under distancing; above it, more.

In [ ]:
fig = wf.plot_ribbons(post_runs, "I", q=(0.025, 0.5, 0.975))
fig.add_vline(x=35, line_dash="dot", annotation_text="distancing starts")
fig.update_layout(xaxis_title="day")
fig.show()

fig = wf.plot_scenarios(post_runs, "I", ref="baseline")
fig.add_vline(x=35, line_dash="dot", annotation_text="distancing starts")
fig.update_layout(xaxis_title="day")
fig.show()

diff = post_runs.difference("distancing", "baseline", "I")
before = np.abs(diff[:, post_runs.times < 35.0]).max()
person_days = -diff.sum(axis=1) * 0.5
pd.Series(
    {
        "largest |difference| before day 35": float(before),
        "draws where distancing lowers the peak": f"{int((post_runs.samples('distancing', 'I').max(axis=1) < post_runs.samples('baseline', 'I').max(axis=1)).sum())} of {post_runs.n}",
        "draws where distancing later has more infectious people": f"{int((diff.max(axis=1) > 0).sum())} of {post_runs.n}",
        "infectious person-days averted, 5%": round(float(np.quantile(person_days, 0.05))),
        "infectious person-days averted, median": round(float(np.quantile(person_days, 0.5))),
        "infectious person-days averted, 95%": round(float(np.quantile(person_days, 0.95))),
    },
    name="E",
    dtype=object,
)

> **What to check**
>
> - First figure: two colours, *baseline* and *distancing*, identical up to the dotted line; after it the distancing ribbon is lower.
> - Second figure: the band sits exactly on zero up to day 35, then drops well **below** the dashed zero line (fewer infectious people); in the weeks after, the upper part of the band rises **above** zero — for many draws distancing flattens and delays the epidemic rather than only shrinking it. The legend reads *distancing − baseline*.
> - Table: the difference before day 35 is 0; distancing lowers the peak in most draws (not in those whose epidemic had already peaked by day 35); many draws have more infectious people later on; the infectious person-days averted are positive at the 5% quantile but range over several-fold — how much distancing helps depends on which of the good fits is right.

## F. The figures compose, and stand for a few lines

One canvas, three layers: the posterior 95% band, the sixteen optimised fits
as spaghetti, and the data. Then the same scenario figure from the
**optimised points** instead of the posterior, with each fit's own effect of
distancing added as a line from `fitted_runs.difference(...)` — the data
behind every figure is public, so a layer the functions do not draw is a few
lines of your own Plotly. The table checks that the figures draw exactly that
public data: the band against `post_runs.quantiles()`, the spaghetti against
`fitted_runs.samples()`, and the scenario median against the median of
`difference()`.

In [ ]:
fig = wf.plot_ribbons(post_runs, "I", q=(0.025, 0.5, 0.975), scenarios=["baseline"], title="F. Posterior band, optimised fits and data on one figure")
wf.plot_spaghetti(fitted_runs, "I", n=None, fig=fig, name="optimised fits", opacity=0.8)
wf.add_targets(fig, bm.targets, name="data")
fig.update_layout(xaxis_title="day")
fig.update_yaxes(type="log", range=[1, 6])
fig.show()

fig = wf.plot_scenarios(fitted_runs, "I", q=(0.025, 0.5, 0.975), title="F. Effect of distancing for each optimised fit")
for i, path in enumerate(fitted_runs.difference("distancing", "baseline", "I")):
    fig.add_scatter(x=fitted_runs.times, y=path, mode="lines", line={"width": 1, "dash": "dot"}, name=f"fit {i}", showlegend=i == 0, legendgroup="fits")
fig.update_layout(xaxis_title="day")
fig.show()

ribbon = wf.plot_ribbons(post_runs, "I", scenarios=["baseline"])
frame = post_runs.quantiles()["baseline"]
t = len(post_runs.times)
spaghetti = wf.plot_spaghetti(fitted_runs, "I", n=None)
lines = np.asarray(spaghetti.data[0].y, dtype=float).reshape(fitted_runs.n, t + 1)[:, :t]
scenario_fig = wf.plot_scenarios(post_runs, "I")
pd.Series(
    {
        "max |outer band - quantiles() 2.5%|": float(np.max(np.abs(np.asarray(ribbon.data[0].y[:t]) - frame[("I", "0.025")].to_numpy()))),
        "max |median line - quantiles() 50%|": float(np.max(np.abs(np.asarray(ribbon.data[2].y) - frame[("I", "0.5")].to_numpy()))),
        "max |spaghetti - samples()|": float(np.max(np.abs(lines - fitted_runs.samples("baseline", "I")))),
        "max |scenario median - median(difference())|": float(np.max(np.abs(np.asarray(scenario_fig.data[2].y) - np.median(post_runs.difference("distancing", "baseline", "I"), axis=0)))),
    },
    name="F",
)

> **What to check**
>
> - First figure: the kept title, the posterior band and median, the optimised fits drawn over it, and the data dots — one legend entry per layer. The fits lie inside the band through the data and spread across it afterwards.
> - Second figure: the dotted per-fit lines all sit on zero until day 35, then fall below it to clearly different depths (and some rebound above it later); the band summarises them.
> - Every number in the table is 0 (or float rounding).